# TRM Search Controller — Training Notebook
Run each cell in order. All outputs and checkpoints are saved to Google Drive so nothing is lost when the session ends.

## 1. Mount Drive & set working directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORKDIR = '/content/drive/MyDrive/Gen_TRM'
os.chdir(WORKDIR)
print('Working directory:', os.getcwd())

## 2. Install dependencies
Colab already has PyTorch — only install the extras.

In [ ]:
!pip install -q adam-atan2 pydantic argdantic omegaconf hydra-core huggingface_hub coolname einops

## 3. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 4. Load TRM checkpoint

In [ ]:
from load_trm import load_trm, load_identifier_map

CHECKPOINT = 'weights/trm_arc_v1/arc_v1_public/step_518071'

trm, cfg = load_trm(CHECKPOINT, device=str(DEVICE))
id_map = load_identifier_map()
print(f'Loaded. hidden_size={cfg["hidden_size"]}, seq_len={cfg["seq_len"]}')

## 5. Build training batches from evaluation puzzles
We train on evaluation puzzles — these are the ones the TRM struggles with, giving the controller real learning signal.

In [ ]:
import json
import numpy as np
from test_arc_puzzle import encode_grid

def make_batch(puzzle_id, test_input, device):
    num_id = id_map.get(puzzle_id, 0)
    tokens = torch.tensor(encode_grid(test_input), dtype=torch.int32, device=device)
    return {
        'inputs': tokens.unsqueeze(0),
        'puzzle_identifiers': torch.tensor([num_id], dtype=torch.int32, device=device),
    }

with open('kaggle/combined/arc-agi_evaluation_challenges.json') as f:
    challenges = json.load(f)
with open('kaggle/combined/arc-agi_evaluation_solutions.json') as f:
    solutions = json.load(f)

batches = []
puzzle_ids = []
for puzzle_id, puzzle in challenges.items():
    test_input = puzzle['test'][0]['input']
    batches.append(make_batch(puzzle_id, test_input, DEVICE))
    puzzle_ids.append(puzzle_id)

print(f'Built {len(batches)} batches from evaluation set')

## 6. Train the controller

In [ ]:
from adaptive_search import train, TrainConfig

config = TrainConfig(
    # Search
    budget_segments=16,
    max_frontier=8,
    branch_m=2,
    cost_expand=1.0,
    lambda_cost=0.05,   # low penalty — we want it to search freely at first
    noise_scale=0.1,

    # Q-learning
    gamma=0.99,
    batch_size=32,
    replay_capacity=10_000,
    learning_rate=3e-4,
    train_every=4,

    # Exploration
    epsilon_start=0.5,
    epsilon_end=0.05,
    epsilon_decay_episodes=500,

    # Architecture — must match TRM
    trm_hidden_size=cfg['hidden_size'],  # 512
    controller_hidden_dim=256,

    # Run
    n_episodes=2000,
    log_every=50,
)

controller = train(trm.inner, batches=batches, config=config, device=DEVICE)

## 7. Save controller checkpoint

In [ ]:
os.makedirs('weights/controller', exist_ok=True)
torch.save(controller.state_dict(), 'weights/controller/v1.pt')
print('Saved to weights/controller/v1.pt')

## 8. Evaluate controller vs greedy baseline

In [ ]:
from adaptive_search import greedy_search, collect_rollout, TrainConfig
from test_arc_puzzle import decode_output

controller.eval()

greedy_correct = 0
controller_correct = 0
n_eval = min(50, len(batches))

for i in range(n_eval):
    puzzle_id = puzzle_ids[i]
    batch = batches[i]
    gold = np.array(solutions[puzzle_id][0], dtype=np.int32)

    # Greedy baseline (8 steps)
    g = greedy_search(trm.inner, batch, num_steps=8)
    pred_g = decode_output(g.output[0].cpu().numpy(), gold.shape)
    greedy_correct += int((pred_g == gold).all())

    # Controller search (budget=16)
    _, best_c = collect_rollout(
        trm=trm.inner,
        controller=controller,
        batch=batch,
        budget_segments=16,
        max_frontier=config.max_frontier,
        branch_m=config.branch_m,
        epsilon=0.0,   # greedy at eval time
    )
    pred_c = decode_output(best_c.output[0].cpu().numpy(), gold.shape)
    controller_correct += int((pred_c == gold).all())

print(f'Greedy (8 steps):     {greedy_correct}/{n_eval} = {greedy_correct/n_eval*100:.1f}%')
print(f'Controller (budget=16): {controller_correct}/{n_eval} = {controller_correct/n_eval*100:.1f}%')

## 9. Resume training (run this cell if session expired)
Loads the saved controller and continues training from where it left off.

In [ ]:
from adaptive_search import SearchController

controller = SearchController(trm_hidden_size=cfg['hidden_size'], hidden_dim=256).to(DEVICE)
controller.load_state_dict(torch.load('weights/controller/v1.pt', map_location=DEVICE))
print('Controller loaded, resuming training...')

config.n_episodes = 1000   # additional episodes
config.epsilon_start = 0.2  # lower epsilon since already partially trained

controller = train(trm.inner, batches=batches, config=config, device=DEVICE,
                   controller=controller)